In [1]:
import os
from pyspark.sql import SparkSession, DataFrame
from pyspark import SparkConf
from pyspark.sql.functions import (
    col,
    when,
    trim,
    to_date,
    regexp_extract,
    regexp_replace,
    coalesce,
    split,
    size
)

In [2]:
def create_spark_configuration() -> SparkConf:
    """
    Создает и конфигурирует экземпляр SparkConf для приложения Spark.

    Returns:
        SparkConf: Настроенный экземпляр SparkConf.
    """
    # Получаем имя пользователя
    user_name = os.getenv("USER")

    conf = SparkConf()
    conf.setAppName("lab 1 Glassdoor")
    conf.setMaster("yarn")
    conf.set("spark.submit.deployMode", "client")
    conf.set("spark.executor.memory", "12g")
    conf.set("spark.executor.cores", "8")
    conf.set("spark.executor.instances", "2")
    conf.set("spark.driver.memory", "4g")
    conf.set("spark.driver.cores", "2")
    conf.set("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.0")
    conf.set("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    conf.set("spark.sql.catalog.spark_catalog", "org.apache.iceberg.spark.SparkCatalog")
    conf.set("spark.sql.catalog.spark_catalog.type", "hadoop")
    conf.set("spark.sql.catalog.spark_catalog.warehouse", f"hdfs:///user/{user_name}/warehouse")
    conf.set("spark.sql.catalog.spark_catalog.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO")

    return conf

In [3]:
conf = create_spark_configuration()

In [4]:
spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark

:: loading settings :: url = jar:file:/opt/spark-3.5.2-bin-hadoop3/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/user0/.ivy2/cache
The jars for the packages stored in: /home/user0/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a28e6413-1796-4a72-ad43-38f9658e1f56;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.0 in central
:: resolution report :: resolve 588ms :: artifacts dl 27ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spa

In [5]:
user_name = os.getenv("USER")
path = f"hdfs:///user/{user_name}/datasets/orlov/all_reviews.csv"
path

'hdfs:///user/user0/datasets/orlov/all_reviews.csv'

In [6]:
df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("multiLine", "true")
    .option("quote", "\"")
    .option("escape", "\"")
    .load(path)
)

In [7]:
df.show(5)

+------+--------------------+--------------------+--------------------+--------------------+------+---------+------------+----------------+--------------------+-------------------------+-----------------+-----------------+----------------+---------------------+--------------------+-------------+--------------------+-----+
|rating|               title|              status|                pros|                cons|advice|Recommend|CEO Approval|Business Outlook|Career Opportunities|Compensation and Benefits|Senior Management|Work/Life Balance|Culture & Values|Diversity & Inclusion|           firm_link|         date|                 job|index|
+------+--------------------+--------------------+--------------------+--------------------+------+---------+------------+----------------+--------------------+-------------------------+-----------------+-----------------+----------------+---------------------+--------------------+-------------+--------------------+-----+
|   5.0|                Good

In [8]:
df.printSchema()

root
 |-- rating: string (nullable = true)
 |-- title: string (nullable = true)
 |-- status: string (nullable = true)
 |-- pros: string (nullable = true)
 |-- cons: string (nullable = true)
 |-- advice: string (nullable = true)
 |-- Recommend: string (nullable = true)
 |-- CEO Approval: string (nullable = true)
 |-- Business Outlook: string (nullable = true)
 |-- Career Opportunities: string (nullable = true)
 |-- Compensation and Benefits: string (nullable = true)
 |-- Senior Management: string (nullable = true)
 |-- Work/Life Balance: string (nullable = true)
 |-- Culture & Values: string (nullable = true)
 |-- Diversity & Inclusion: string (nullable = true)
 |-- firm_link: string (nullable = true)
 |-- date: string (nullable = true)
 |-- job: string (nullable = true)
 |-- index: string (nullable = true)



In [9]:
COLUMN_NAMES = {
    "rating": "overall_rating",
    "title": "headline",
    "status": "current",
    "pros": "pros",
    "cons": "cons",
    "advice": "advice",
    "Recommend": "recommend",
    "CEO Approval": "ceo_approv",
    "Business Outlook": "outlook",
    "Career Opportunities": "career_opp",
    "Compensation and Benefits": "comp_benefits",
    "Senior Management": "senior_mgmt",
    "Work/Life Balance": "work_life_balance",
    "Culture & Values": "culture_values",
    "Diversity & Inclusion": "diversity_inclusion",
    "firm_link": "firm_link",
    "date": "date_review",
    "job": "job_title",
}

missing_columns = [c for c in COLUMN_NAMES if c not in df.columns]
extra_columns = [c for c in df.columns if c not in COLUMN_NAMES]

print("Отсутствуют в файле:", missing_columns or "нет")
print("Не используются:   ", extra_columns or "нет")

Отсутствуют в файле: нет
Не используются:    ['index']


In [10]:
df = df.select(
    *[col(f"`{old}`").alias(new) for old, new in COLUMN_NAMES.items()]
)
df.printSchema()

root
 |-- overall_rating: string (nullable = true)
 |-- headline: string (nullable = true)
 |-- current: string (nullable = true)
 |-- pros: string (nullable = true)
 |-- cons: string (nullable = true)
 |-- advice: string (nullable = true)
 |-- recommend: string (nullable = true)
 |-- ceo_approv: string (nullable = true)
 |-- outlook: string (nullable = true)
 |-- career_opp: string (nullable = true)
 |-- comp_benefits: string (nullable = true)
 |-- senior_mgmt: string (nullable = true)
 |-- work_life_balance: string (nullable = true)
 |-- culture_values: string (nullable = true)
 |-- diversity_inclusion: string (nullable = true)
 |-- firm_link: string (nullable = true)
 |-- date_review: string (nullable = true)
 |-- job_title: string (nullable = true)



In [11]:
spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")

In [12]:
RATING_COLUMNS = [
    "work_life_balance", "culture_values", "diversity_inclusion",
    "career_opp", "comp_benefits", "senior_mgmt"
]
OPINION_COLUMNS = ["recommend", "ceo_approv", "outlook"]
TEXT_COLUMNS = ["headline", "pros", "cons", "advice"]


def blank_to_null(column_name: str):
    """
    Возвращает выражение, в котором пустые строки и строки из пробелов
    заменены на NULL, а у остальных строк удалены крайние пробелы.
    """
    trimmed = trim(col(column_name))
    return when(trimmed == "", None).otherwise(trimmed)


def decode_opinion(column_name: str):
    """
    Возвращает выражение, заменяющее коды v / r / x / o на названия категорий.
    Значения, не совпадающие ни с одним кодом, остаются без изменений.
    """
    c = col(column_name)
    return (
        when(c == "v", "Positive")
        .when(c == "r", "Mild")
        .when(c == "x", "Negative")
        .when(c == "o", "No opinion")
        .otherwise(c)
    )


def word_count(column_name: str):
    """
    Возвращает выражение для количества слов в тексте (NULL для пустого текста).
    """
    c = col(column_name)
    return when(c.isNull(), None).otherwise(size(split(c, r"\s+")))


def transform_dataframe(data: DataFrame, keep_text: bool = False) -> DataFrame:
    """
    Преобразует столбцы DataFrame в нужные типы данных и
    выполняет разбор строковых полей.

    Args:
        data (DataFrame): Исходный DataFrame.
        keep_text (bool): Сохранять ли исходные тексты отзывов.

    Returns:
        DataFrame: Преобразованный DataFrame.
    """
    # Пустые строки -> NULL
    for column_name in data.columns:
        data = data.withColumn(column_name, blank_to_null(column_name))

    # Название компании из ссылки на её страницу
    firm_slug = regexp_extract(col("firm_link"), r"Reviews/(.+?)-Reviews-E\d+", 1)
    data = data.withColumn(
        "firm",
        when(firm_slug == "", None).otherwise(regexp_replace(firm_slug, "-", " "))
    ).drop("firm_link")

    # Дата отзыва: основной формат "Nov 19, 2022", запасной "2022-11-19"
    data = data.withColumn(
        "date_review",
        coalesce(to_date(col("date_review"), "MMM d, yyyy"),
                   to_date(col("date_review"), "yyyy-MM-dd"))
    )

    # Оценки
    data = data.withColumn("overall_rating",
                           col("overall_rating").cast("Double").cast("Integer"))
    for column_name in RATING_COLUMNS:
        data = data.withColumn(column_name, col(column_name).cast("Double"))

    # Расшифровка кодов мнений
    for column_name in OPINION_COLUMNS:
        data = data.withColumn(column_name, decode_opinion(column_name))

    # Разбор строки current
    status = regexp_extract(col("current"), r"^(Current|Former)", 1)
    data = data.withColumn("employee_status",
                           when(status == "", None).otherwise(status))

    tenure_kind = regexp_extract(col("current"), r"(more|less) than (\d+) year", 1)
    tenure_value = regexp_extract(col("current"), r"(more|less) than (\d+) year", 2)
    data = data.withColumn(
        "tenure_years",
        when(tenure_kind == "more", tenure_value.cast("Integer"))
        .when(tenure_kind == "less", 0)
        .otherwise(None)
    )

    # Длина текстов в словах
    for column_name in TEXT_COLUMNS:
        data = data.withColumn(f"{column_name}_words", word_count(column_name))

    if not keep_text:
        data = data.drop(*TEXT_COLUMNS)

    # Удобный порядок столбцов: сначала компания, дата и автор отзыва
    first = ["firm", "date_review", "job_title", "current",
             "employee_status", "tenure_years", "overall_rating"]
    return data.select(*first, *[c for c in data.columns if c not in first])
#%%


In [13]:
df = transform_dataframe(df)

In [14]:
df.show(5)

+--------------------+-----------+--------------------+--------------------+---------------+------------+--------------+----------+----------+----------+----------+-------------+-----------+-----------------+--------------+-------------------+--------------+----------+----------+------------+
|                firm|date_review|           job_title|             current|employee_status|tenure_years|overall_rating| recommend|ceo_approv|   outlook|career_opp|comp_benefits|senior_mgmt|work_life_balance|culture_values|diversity_inclusion|headline_words|pros_words|cons_words|advice_words|
+--------------------+-----------+--------------------+--------------------+---------------+------------+--------------+----------+----------+----------+----------+-------------+-----------+-----------------+--------------+-------------------+--------------+----------+----------+------------+
|Baja Steel and Fence| 2022-11-19|      Manager Design|Current Employee,...|        Current|          10|             

In [15]:
df.printSchema()

root
 |-- firm: string (nullable = true)
 |-- date_review: date (nullable = true)
 |-- job_title: string (nullable = true)
 |-- current: string (nullable = true)
 |-- employee_status: string (nullable = true)
 |-- tenure_years: integer (nullable = true)
 |-- overall_rating: integer (nullable = true)
 |-- recommend: string (nullable = true)
 |-- ceo_approv: string (nullable = true)
 |-- outlook: string (nullable = true)
 |-- career_opp: double (nullable = true)
 |-- comp_benefits: double (nullable = true)
 |-- senior_mgmt: double (nullable = true)
 |-- work_life_balance: double (nullable = true)
 |-- culture_values: double (nullable = true)
 |-- diversity_inclusion: double (nullable = true)
 |-- headline_words: integer (nullable = true)
 |-- pros_words: integer (nullable = true)
 |-- cons_words: integer (nullable = true)
 |-- advice_words: integer (nullable = true)



In [16]:
database_name = f"orlov_database"
table_name = "glassdoor_reviews"
database_name

'orlov_database'

In [17]:
create_database_sql = f"""
CREATE DATABASE IF NOT EXISTS spark_catalog.{database_name}
"""

In [18]:
spark.sql(create_database_sql)

DataFrame[]

In [19]:
spark.catalog.setCurrentDatabase(database_name)

In [20]:
(
    df
    .repartition(16)
    .writeTo(table_name)
    .using("iceberg")
    .createOrReplace()
)

In [21]:
for table in spark.catalog.listTables():
    print(table.name)

glassdoor_reviews
sobd_lab1_table
sobd_lab1_processed_table


In [22]:
spark.table(table_name).count()

9901889

In [23]:
spark.stop()